In [2]:
import pandas as pd
import numpy as np
import sys

## Reading Text Files in Pieces

For large files it is often impractical to load everything into memory at once. pandas provides two approaches:

### Read a Fixed Number of Rows — `nrows`

Use `nrows` to read only the first `n` rows without scanning the rest of the file:

```python
pd.read_csv(file, nrows=5)
```

Useful for inspecting the structure of a large file before committing to a full read.

### Iterate in Chunks — `chunksize`

Use `chunksize` to get a `TextFileReader` object that yields the file in pieces:

```python
chunker = pd.read_csv(file, chunksize=1000)
for piece in chunker:
    ...  # each piece is a DataFrame of chunksize rows
```

- `TextFileReader` is iterable — each iteration yields one chunk as a DataFrame.
- `TextFileReader.get_chunk(n)` reads the next `n` rows on demand.
- Combine `fill_value=0` with `Series.add` to safely accumulate results across chunks.

### Display Tip

Set `pd.options.display.max_rows` to a small number to keep large DataFrame output compact:

```python
pd.options.display.max_rows = 10
```

## Key Points

- `nrows=n` reads only the first `n` rows — efficient for previewing large files.
- `chunksize=n` returns a `TextFileReader` that iterates over the file `n` rows at a time.
- Each chunk is a regular DataFrame — all normal pandas operations apply.
- Use `Series.add(..., fill_value=0)` to accumulate statistics across chunks without `NaN` propagation.

In [3]:
pd.options.display.max_rows = 10

result = pd.read_csv("../examples/ex6.csv")

result

,one,two,three,four,key
0,0.467976,-0.038649,-0.295344,-1.824726,L
1,-0.358893,1.404453,0.704965,-0.200638,B
2,-0.501840,0.659254,-0.421691,-0.057688,G
3,0.204886,1.074134,1.388361,-0.982404,R
4,0.354628,-0.133116,0.283763,-0.837063,Q
...,...,...,...,...,...
9995,2.311896,-0.417070,-1.409599,-0.515821,L
9996,-0.479893,-0.650419,0.745152,-0.646038,E
9997,0.523331,0.787112,0.486066,1.093156,K
9998,-0.362559,0.598894,-1.843201,0.887292,G


In [4]:
pd.read_csv("../examples/ex6.csv", nrows=5)

,one,two,three,four,key
0,0.467976,-0.038649,-0.295344,-1.824726,L
1,-0.358893,1.404453,0.704965,-0.200638,B
2,-0.501840,0.659254,-0.421691,-0.057688,G
3,0.204886,1.074134,1.388361,-0.982404,R
4,0.354628,-0.133116,0.283763,-0.837063,Q


In [5]:
chunker = pd.read_csv("../examples/ex6.csv", chunksize=1000)

type(chunker)

pandas.io.parsers.readers.TextFileReader

In [6]:
# Aggregate value counts across chunks
chunker = pd.read_csv("../examples/ex6.csv", chunksize=1000)

tot = pd.Series([], dtype="int64")
for piece in chunker:
    tot = tot.add(piece["key"].value_counts(), fill_value=0)

tot = tot.sort_values(ascending=False)

tot[:10]

key
E    368.0
X    364.0
L    346.0
O    343.0
Q    340.0
M    338.0
J    337.0
F    335.0
K    334.0
H    330.0
dtype: float64

## Writing Data to Text Format

DataFrame's `to_csv` method writes data to a delimited text file.

### Default Behaviour

- Delimiter is `,`.
- Both the row index and column headers are written.
- Missing values (`NaN`) appear as **empty strings**.

### Common Options

| Argument | Effect |
|----------|--------|
| `sep="|"` | Use a different delimiter |
| `na_rep="NULL"` | Replace `NaN` with a custom string |
| `index=False` | Omit the row index |
| `header=False` | Omit the column header row |
| `columns=[...]` | Write only the specified columns, in that order |

Pass `sys.stdout` as the path to print the CSV output to the console instead of writing a file — useful for inspecting the result.

## Key Points

- `to_csv` writes to a file path or any file-like object (e.g. `sys.stdout`).
- `NaN` becomes an empty string by default; use `na_rep=` to change it.
- `index=False` and `header=False` suppress the index and header respectively.
- `columns=[...]` selects and reorders which columns are written.

In [7]:
data = pd.read_csv("../examples/ex5.csv")

data

,something,a,b,c,d,message
0,one,1,2,3.0,4,NaN
1,two,5,6,NaN,8,world
2,three,9,10,11.0,12,foo


In [8]:
# Write to file
data.to_csv("../examples/out.csv")

In [9]:
# Custom delimiter
data.to_csv(sys.stdout, sep="|")

|something|a|b|c|d|message
0|one|1|2|3.0|4|
1|two|5|6||8|world
2|three|9|10|11.0|12|foo


In [10]:
# Custom NA representation
data.to_csv(sys.stdout, na_rep="NULL")

,something,a,b,c,d,message
0,one,1,2,3.0,4,NULL
1,two,5,6,NULL,8,world
2,three,9,10,11.0,12,foo


In [11]:
# Suppress index and header
data.to_csv(sys.stdout, index=False, header=False)

one,1,2,3.0,4,
two,5,6,,8,world
three,9,10,11.0,12,foo


In [12]:
# Write a subset of columns in a chosen order
data.to_csv(sys.stdout, index=False, columns=["a", "b", "c"])

a,b,c
1,2,3.0
5,6,
9,10,11.0
